# Hands-on Lab: Build, Evaluate & Optimize AI Functions in Snowflake
**Use case:** Support-ticket triage with business rules.

In ~30 minutes you will:
1. **Load** a small labeled dataset (40 tickets)
2. **Baseline** a built-in function (`AI_CLASSIFY`) and **evaluate** it with an `AI_FUNCTION_EVALUATION` experiment
3. **Create** a custom function with `CREATE AI FUNCTION` (deliberately small model, simple prompt)
4. **Evaluate** the custom function
5. **Optimize** it with an `AI_FUNCTION_OPTIMIZATION` experiment (prompt/body tuning + model comparison)
6. **Compare** scores and deploy the winning body

Docs: [AI function evaluation](https://docs.snowflake.com/en/sql-reference/functions/ai_function_evaluation) · [AI function optimization](https://docs.snowflake.com/en/sql-reference/functions/ai_function_optimization) · [CREATE AI FUNCTION](https://docs.snowflake.com/en/sql-reference/sql/create-ai-function) · [EXPERIMENT](https://docs.snowflake.com/en/sql-reference/sql/experiment) · [AI Function Studio](https://docs.snowflake.com/en/user-guide/snowflake-cortex/ai-function-studio)

## Step 0 — Setup

In [ ]:
%%sql -r setup
USE ROLE ACCOUNTADMIN;
CREATE DATABASE IF NOT EXISTS VHOL_AI_DB;
CREATE SCHEMA IF NOT EXISTS VHOL_AI_DB.AI_FUNC;
USE SCHEMA VHOL_AI_DB.AI_FUNC;

# Step 1 — Labeled data
Notice the tricky tickets at the bottom: they *mention* billing or cancelling but belong elsewhere. Our routing **rules**:
- A ticket is `CANCELLATION` only if the customer wants to end the **whole** account/contract. Dropping an add-on is `BILLING`.
- Threats to cancel don't change the category — route by the actual problem.
- Broken billing pages/links are `TECHNICAL`; can't log in to see a bill is `ACCOUNT_ACCESS`.

In [ ]:
%%sql -r load_data
CREATE OR REPLACE TABLE SUPPORT_TICKETS (TICKET_ID INT, TICKET_TEXT VARCHAR, CATEGORY VARCHAR) AS
SELECT * FROM VALUES
  (1, 'I was charged twice for my October invoice, please refund the duplicate.', 'BILLING'),
  (2, 'Our credit card on file expired, how do I update payment details?', 'BILLING'),
  (3, 'Why did my monthly bill jump from $49 to $99 without notice?', 'BILLING'),
  (4, 'Can I get a copy of last quarter''s invoices for our auditors?', 'BILLING'),
  (5, 'We were billed for 50 seats but only have 32 users.', 'BILLING'),
  (6, 'Please switch us from monthly to annual billing to get the discount.', 'BILLING'),
  (7, 'The dashboard throws a 500 error every time I export to CSV.', 'TECHNICAL'),
  (8, 'API calls to /v2/orders started timing out after your last release.', 'TECHNICAL'),
  (9, 'The mobile app crashes on launch on Android 15.', 'TECHNICAL'),
  (10, 'Webhooks are firing twice for every event.', 'TECHNICAL'),
  (11, 'Reports load extremely slowly, over 2 minutes for a 1k-row table.', 'TECHNICAL'),
  (12, 'SSO login worked yesterday but now SAML returns an invalid signature error.', 'TECHNICAL'),
  (13, 'I forgot my password and the reset email never arrives.', 'ACCOUNT_ACCESS'),
  (14, 'My account got locked after too many login attempts.', 'ACCOUNT_ACCESS'),
  (15, 'Our admin left the company, how do we transfer ownership of the workspace?', 'ACCOUNT_ACCESS'),
  (16, 'I need to add two new users but the invite button is greyed out for me.', 'ACCOUNT_ACCESS'),
  (17, 'I lost my phone and can''t get past two-factor authentication.', 'ACCOUNT_ACCESS'),
  (18, 'How do I change the email address I use to sign in?', 'ACCOUNT_ACCESS'),
  (19, 'It would be great if you supported dark mode.', 'FEATURE_REQUEST'),
  (20, 'Could you add a Slack integration for alert notifications?', 'FEATURE_REQUEST'),
  (21, 'Please let us schedule reports to be emailed every Monday.', 'FEATURE_REQUEST'),
  (22, 'We''d love bulk-edit for tags instead of one at a time.', 'FEATURE_REQUEST'),
  (23, 'Any plans to support exporting to Parquet?', 'FEATURE_REQUEST'),
  (24, 'A Spanish-language UI would really help our LATAM team.', 'FEATURE_REQUEST'),
  (25, 'We''d like to cancel our subscription at the end of this term.', 'CANCELLATION'),
  (26, 'Please close our account, we''re moving to another vendor.', 'CANCELLATION'),
  (27, 'How do I downgrade to free and stop all future charges? We''re winding down the project.', 'CANCELLATION'),
  (28, 'Do not renew our contract next month.', 'CANCELLATION'),
  (29, 'We''re shutting down the team that used this; terminate the account please.', 'CANCELLATION'),
  (30, 'I want out. Your product has been nothing but trouble, end it today.', 'CANCELLATION'),
  (31, 'Since the last update the invoice PDF download link is broken.', 'TECHNICAL'),
  (32, 'If you don''t fix the export bug I''ll cancel, but for now just fix it please.', 'TECHNICAL'),
  (33, 'I can''t log in to view my bill.', 'ACCOUNT_ACCESS'),
  (34, 'Feature idea: let us pay invoices with ACH instead of card.', 'FEATURE_REQUEST'),
  (35, 'Refund me for the month the service was down or I''m leaving.', 'BILLING'),
  (36, 'The billing page shows a spinner forever and never loads.', 'TECHNICAL'),
  (37, 'We''d like to stop paying for the analytics add-on but keep the core plan.', 'BILLING'),
  (38, 'Can you reactivate my deactivated user so I can log in again?', 'ACCOUNT_ACCESS'),
  (39, 'Please remove the auto-renew so our account ends on Dec 31.', 'CANCELLATION'),
  (40, 'Would be helpful to see usage per user on the invoice.', 'FEATURE_REQUEST');

SELECT CATEGORY, COUNT(*) AS N FROM SUPPORT_TICKETS GROUP BY 1 ORDER BY 1;

### Wrap the table as a versioned DATASET
Experiments read a **DATASET** version (an immutable snapshot), not a raw table. Version names are case-sensitive.

In [ ]:
%%sql -r dataset
CREATE OR REPLACE DATASET VHOL_AI_DB.AI_FUNC.TICKETS_DS;
ALTER DATASET VHOL_AI_DB.AI_FUNC.TICKETS_DS ADD VERSION 'v1' FROM (SELECT TICKET_TEXT, CATEGORY FROM SUPPORT_TICKETS);

# Step 2 — Built-in baseline: `AI_CLASSIFY`

In [ ]:
%%sql -r builtin_preview
SELECT TICKET_TEXT, CATEGORY AS EXPECTED,
       AI_CLASSIFY(TICKET_TEXT, ['BILLING','TECHNICAL','ACCOUNT_ACCESS','FEATURE_REQUEST','CANCELLATION']):labels[0]::VARCHAR AS PREDICTED
FROM SUPPORT_TICKETS ORDER BY TICKET_ID DESC LIMIT 10;

### Evaluate the built-in call
An evaluation experiment takes a `query_text` (any SQL expression over dataset columns), a metric, and the dataset. `EXECUTE EXPERIMENT` runs **asynchronously**.

In [ ]:
%%sql -r eval_builtin
DROP EXPERIMENT IF EXISTS EVAL_BUILTIN_CLASSIFY;
CREATE EXPERIMENT EVAL_BUILTIN_CLASSIFY
  TYPE = 'AI_FUNCTION_EVALUATION'
  FROM SPECIFICATION $$
    query_text: AI_CLASSIFY(TICKET_TEXT, ['BILLING','TECHNICAL','ACCOUNT_ACCESS','FEATURE_REQUEST','CANCELLATION']):labels[0]::VARCHAR
    metrics:
      - name: exact_match
    dataset:
      name: VHOL_AI_DB.AI_FUNC.TICKETS_DS
      version: v1
      ground_truth: CATEGORY
    evaluation:
      num_eval_runs: 1
  $$;
EXECUTE EXPERIMENT EVAL_BUILTIN_CLASSIFY;

> **Note:** `EXECUTE EXPERIMENT` runs **asynchronously** — the evaluation is now running in the background. We'll check its results later in Step 4.
>
> While it runs, let's build a custom `AI FUNCTION` with a deliberate baseline (small model, no business rules in the prompt) so we have a second candidate to evaluate and then **optimize**.

# Step 3 — `CREATE AI FUNCTION`
A custom AI function wraps `AI_COMPLETE` with a system prompt, a user template, and structured JSON output. We start with a **small, cheap model** and a **naive prompt** (no business rules) — the optimizer will improve it.

In [ ]:
CREATE OR REPLACE AI FUNCTION VHOL_AI_DB.AI_FUNC.TRIAGE_TICKET(TICKET_TEXT VARCHAR)
RETURNS VARCHAR
COMMENT = 'Route a support ticket to a queue'
AS
$$
    SELECT AI_COMPLETE(
    model=>'llama3.1-8b',
    messages=>ARRAY_CONSTRUCT(
        OBJECT_CONSTRUCT('role', 'system', 'content', 'You are a customer support triage assistant for a SaaS company. Classify the support ticket into exactly one category: BILLING, TECHNICAL, ACCOUNT_ACCESS, FEATURE_REQUEST, or CANCELLATION.'),
        OBJECT_CONSTRUCT('role', 'user', 'content', 'Ticket: ' || TICKET_TEXT)
    ),
    response_format=>PARSE_JSON('{
      "type": "json",
      "schema": {"type": "object",
                 "properties": {"category": {"type": "string", "description": "One of BILLING, TECHNICAL, ACCOUNT_ACCESS, FEATURE_REQUEST, CANCELLATION"}},
                 "required": ["category"], "additionalProperties": false}}')
    ):category::VARCHAR
$$;

In [ ]:
%%sql -r try_fn
SELECT TICKET_TEXT, CATEGORY AS EXPECTED, TRIAGE_TICKET(TICKET_TEXT) AS PREDICTED
FROM SUPPORT_TICKETS ORDER BY TICKET_ID DESC LIMIT 10;

# Step 4 — Evaluate the custom function

In [ ]:
%%sql -r eval_custom
DROP EXPERIMENT IF EXISTS EVAL_TRIAGE_CUSTOM_V1;
CREATE EXPERIMENT EVAL_TRIAGE_CUSTOM_V1
  TYPE = 'AI_FUNCTION_EVALUATION' -- Evaluation experiment
  FROM SPECIFICATION $$
    query_text: VHOL_AI_DB.AI_FUNC.TRIAGE_TICKET(TICKET_TEXT)
    metrics:
      - name: exact_match
    dataset:
      name: VHOL_AI_DB.AI_FUNC.TICKETS_DS
      version: v1
      ground_truth: CATEGORY
    evaluation:
      num_eval_runs: 1
  $$;
EXECUTE EXPERIMENT EVAL_TRIAGE_CUSTOM_V1;

Check status — re-run until both show `FINISHED` (usually 1–3 min). Scores live in `metadata:metrics`.

In [ ]:
%%sql -r eval_status
SHOW RUNS IN EXPERIMENT EVAL_BUILTIN_CLASSIFY;
SELECT 'EVAL_BUILTIN_CLASSIFY' AS EXPERIMENT, "name" AS RUN, PARSE_JSON("metadata"):status::VARCHAR AS STATUS, PARSE_JSON("metadata"):metrics AS METRICS FROM TABLE(RESULT_SCAN(LAST_QUERY_ID()));

In [ ]:
%%sql -r eval_status_custom
SHOW RUNS IN EXPERIMENT EVAL_TRIAGE_CUSTOM_V1;
SELECT 'EVAL_TRIAGE_CUSTOM_V1' AS EXPERIMENT, "name" AS RUN, PARSE_JSON("metadata"):status::VARCHAR AS STATUS, PARSE_JSON("metadata"):metrics AS METRICS FROM TABLE(RESULT_SCAN(LAST_QUERY_ID()));

### Per-row failures (where did the model go wrong?)

In [ ]:
%%sql -r eval_failures
CREATE OR REPLACE TEMPORARY FILE FORMAT EVAL_JSON TYPE = JSON STRIP_OUTER_ARRAY = TRUE;
SELECT $1:input_text::STRING AS INPUT, $1:expected::STRING AS EXPECTED, $1:predicted::STRING AS PREDICTED, $1:metric_feedback::STRING AS FEEDBACK, $1:metric_score::FLOAT AS SCORE
FROM 'snow://experiment/EVAL_TRIAGE_CUSTOM_V1/versions/EVAL_1/eval_detail.json' (FILE_FORMAT => EVAL_JSON)
WHERE $1:metric_score::FLOAT < 1;

# Step 5 — Optimize
The optimizer (GEPA) reflects on failures with a strong `reflection_model`, rewrites the whole function body (prompt, model ref, SQL pre/post-processing), and scores each candidate — **for every model in `models` concurrently**. `ultra-light` (~2 iterations) fits a lab; use `light`/`medium`/`heavy` for real work.

> For our example this can run ~5–10 min.

In [ ]:
%%sql -r optimize
DROP EXPERIMENT IF EXISTS OPT_TRIAGE;
CREATE EXPERIMENT OPT_TRIAGE
  TYPE = 'AI_FUNCTION_OPTIMIZATION' -- Optimization experiment
  FROM SPECIFICATION $$
    function:
      function_name: "VHOL_AI_DB.AI_FUNC.TRIAGE_TICKET(VARCHAR)"
    metrics:
      - name: exact_match
    dataset:
      name: VHOL_AI_DB.AI_FUNC.TICKETS_DS
      version: v1
      column_mapping:
        argument_mapping:
          TICKET_TEXT: TICKET_TEXT
        ground_truth: CATEGORY
    optimization:
      models: [llama3.1-8b, claude-haiku-4-5, openai-gpt-5.1]
      reflection_model: claude-opus-4-7
      strategy: quality_first
      budget: ultra-light
  $$;
EXECUTE EXPERIMENT OPT_TRIAGE;

Monitor: runs are `SEED`, `ITER_1..N`. Re-run below cell until all are `FINISHED`.
- `val_score` — exact-match accuracy on the validation split
- `cost_compared_to_seed` — relative cost (1.0 = seed)
- `is_frontier = 1` — on the cost/quality Pareto frontier

In [ ]:
%%sql -r dataframe_3
SHOW RUNS IN EXPERIMENT OPT_TRIAGE;
-- SHOW RUN PARAMETERS IN EXPERIMENT OPT_TRIAGE RUN SEED;

In [ ]:
%%sql -r opt_status
EXECUTE IMMEDIATE $$
DECLARE res RESULTSET;
BEGIN
  CREATE OR REPLACE TEMPORARY TABLE _OPT_RUNS (
    RUN VARCHAR, STATUS VARCHAR, VAL_SCORE FLOAT, COST_VS_SEED FLOAT, IS_FRONTIER INT, MODEL VARCHAR, PROMPT_PREVIEW VARCHAR
  );
  SHOW RUNS IN EXPERIMENT OPT_TRIAGE;
  INSERT INTO _OPT_RUNS (RUN, STATUS, VAL_SCORE, COST_VS_SEED, IS_FRONTIER)
    SELECT "name",
           PARSE_JSON("metadata"):status::VARCHAR,
           PARSE_JSON("metadata"):metrics:val_score:value::FLOAT,
           PARSE_JSON("metadata"):metrics:cost_compared_to_seed:value::FLOAT,
           PARSE_JSON("metadata"):metrics:is_frontier:value::INT
    FROM TABLE(RESULT_SCAN(LAST_QUERY_ID()));

  SHOW RUN PARAMETERS IN EXPERIMENT OPT_TRIAGE;
  LET params_qid VARCHAR := LAST_QUERY_ID();
  CREATE OR REPLACE TEMPORARY TABLE _params_pivot AS
    SELECT "run_name",
           MAX(CASE WHEN "name" = 'model' THEN "value" END) AS model,
           MAX(CASE WHEN "name" = 'function_impl' THEN "value" END) AS prompt_preview
    FROM TABLE(RESULT_SCAN(:params_qid))
    GROUP BY "run_name";

  UPDATE _OPT_RUNS o
  SET o.MODEL = p.model,
      o.PROMPT_PREVIEW = p.prompt_preview
  FROM _params_pivot p
  WHERE o.RUN = p."run_name";

  res := (SELECT * FROM _OPT_RUNS ORDER BY VAL_SCORE DESC NULLS LAST);
  RETURN TABLE(res);
END;
$$;

# Step 6 — Pick the winner & deploy for V2
Runs on the Pareto frontier have `is_frontier = 1`. This cell picks the highest `val_score` (cheapest on ties), prints the optimized body, and deploys it as `TRIAGE_TICKET_V2` (the seed stays in place for A/B).

In [ ]:
%%sql -r deploy_v2
EXECUTE IMMEDIATE $$
DECLARE best_run VARCHAR; best_model VARCHAR; body VARCHAR; res RESULTSET;
BEGIN
  -- 1. Pick the winner: highest val_score, cheapest on ties
  SHOW RUNS IN EXPERIMENT VHOL_AI_DB.AI_FUNC.OPT_TRIAGE;
  SELECT "name" INTO :best_run FROM TABLE(RESULT_SCAN(LAST_QUERY_ID()))
  WHERE PARSE_JSON("metadata"):status = 'FINISHED'
    AND PARSE_JSON("metadata"):metrics:val_score:value IS NOT NULL
    AND PARSE_JSON("metadata"):metrics:cost_compared_to_seed:value IS NOT NULL
  ORDER BY PARSE_JSON("metadata"):metrics:val_score:value::FLOAT DESC NULLS LAST,
           PARSE_JSON("metadata"):metrics:cost_compared_to_seed:value::FLOAT ASC NULLS LAST
  LIMIT 1;
  -- 2. Read its optimized function body + model
  EXECUTE IMMEDIATE 'SHOW RUN PARAMETERS IN EXPERIMENT VHOL_AI_DB.AI_FUNC.OPT_TRIAGE RUN ' || :best_run;
  SELECT MAX(IFF("name"='function_impl', "value", NULL)), MAX(IFF("name"='model', "value", NULL))
    INTO :body, :best_model FROM TABLE(RESULT_SCAN(LAST_QUERY_ID()));
  -- 3. Deploy as V2 (seed stays for A/B)
  EXECUTE IMMEDIATE 'CREATE OR REPLACE AI FUNCTION VHOL_AI_DB.AI_FUNC.TRIAGE_TICKET_V2(TICKET_TEXT VARCHAR) RETURNS VARCHAR AS '
                    || '$' || '$' || :body || '$' || '$';
  res := (SELECT :best_run AS BEST_RUN, :best_model AS MODEL, :body AS OPTIMIZED_BODY);
  RETURN TABLE(res);
END;
$$;

# Evaluate V2 on the same dataset

In [ ]:
%%sql -r eval_v2
DROP EXPERIMENT IF EXISTS EVAL_TRIAGE_V2;
CREATE EXPERIMENT EVAL_TRIAGE_V2
  TYPE = 'AI_FUNCTION_EVALUATION'
  FROM SPECIFICATION $$
    query_text: VHOL_AI_DB.AI_FUNC.TRIAGE_TICKET_V2(TICKET_TEXT)
    metrics:
      - name: exact_match
    dataset:
      name: VHOL_AI_DB.AI_FUNC.TICKETS_DS
      version: v1
      ground_truth: CATEGORY
    evaluation:
      num_eval_runs: 1
  $$;
EXECUTE EXPERIMENT EVAL_TRIAGE_V2;

# Compare all evaluations
> **Tip:** If you just kicked off `EVAL_TRIAGE_V2`, it may take a minute or two before its run appears. Re-run this cell until you see all 3 experiments with `FINISHED` status.

In [ ]:
%%sql -r compare
EXECUTE IMMEDIATE $$
DECLARE
  res RESULTSET;
  c1 CURSOR FOR SELECT column1 AS EXP FROM VALUES ('EVAL_BUILTIN_CLASSIFY'), ('EVAL_TRIAGE_CUSTOM_V1'), ('EVAL_TRIAGE_V2');
  x VARCHAR;
BEGIN
  CREATE OR REPLACE TEMPORARY TABLE EVAL_SCORES (
    EXPERIMENT VARCHAR, RUN VARCHAR, STATUS VARCHAR, EXACT_MATCH FLOAT, MODEL VARCHAR, PROMPT_OR_QUERY VARCHAR
  );
  FOR rec IN c1 DO
    x := rec.EXP;
    EXECUTE IMMEDIATE 'SHOW RUNS IN EXPERIMENT VHOL_AI_DB.AI_FUNC.' || :x;
    INSERT INTO EVAL_SCORES (EXPERIMENT, RUN, STATUS, EXACT_MATCH)
      SELECT :x, "name", PARSE_JSON("metadata"):status::VARCHAR, PARSE_JSON("metadata"):metrics:score:value::FLOAT
      FROM TABLE(RESULT_SCAN(LAST_QUERY_ID()));
  END FOR;

  LET c2 CURSOR FOR SELECT EXPERIMENT, RUN FROM EVAL_SCORES;
  FOR rec IN c2 DO
    BEGIN
      EXECUTE IMMEDIATE 'SHOW RUN PARAMETERS IN EXPERIMENT VHOL_AI_DB.AI_FUNC.' || rec.EXPERIMENT || ' RUN ' || rec.RUN;
      CREATE OR REPLACE TEMPORARY TABLE _eval_params AS
        SELECT * FROM TABLE(RESULT_SCAN(LAST_QUERY_ID()));
      UPDATE EVAL_SCORES SET
        MODEL = COALESCE(
          (SELECT "value" FROM _eval_params WHERE "name" = 'model'),
          'built-in'
        ),
        PROMPT_OR_QUERY = COALESCE(
          (SELECT LEFT("value", 200) FROM _eval_params WHERE "name" = 'function_impl'),
          (SELECT LEFT("value", 200) FROM _eval_params WHERE "name" = 'query_text')
        )
      WHERE EXPERIMENT = rec.EXPERIMENT AND RUN = rec.RUN;
    EXCEPTION WHEN OTHER THEN NULL;
    END;
  END FOR;

  res := (SELECT * FROM EVAL_SCORES ORDER BY EXACT_MATCH DESC NULLS LAST);
  RETURN TABLE(res);
END;
$$;

# Visual comparison - Quality

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd

df = compare.to_pandas() if hasattr(compare, 'to_pandas') else compare.copy()
df = df.dropna(subset=['EXACT_MATCH']).sort_values('EXACT_MATCH', ascending=True)

labels = df['EXPERIMENT'] + '  /  ' + df['RUN']
scores = df['EXACT_MATCH']

palette = {'EVAL_BUILTIN_CLASSIFY': '#A8A8A8', 'EVAL_TRIAGE_CUSTOM_V1': '#FF6F61', 'EVAL_TRIAGE_V2': '#29B5E8'}
colors = [palette.get(e, '#6BCB77') for e in df['EXPERIMENT']]

fig, ax = plt.subplots(figsize=(9, max(3, len(df) * 0.7)))
bars = ax.barh(labels, scores, color=colors, edgecolor='white', height=0.6)
ax.set_xlabel('Exact Match Score')
ax.set_title('Evaluation Comparison')
ax.set_xlim(0, 1.08)
for bar, val in zip(bars, scores):
    ax.text(val + 0.01, bar.get_y() + bar.get_height() / 2, f'{val:.1%}', va='center', fontsize=10)
plt.tight_layout()
plt.show()

# Visual Comparison - Cost + Quality

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
from snowflake.snowpark.context import get_active_session

session = get_active_session()

# --- Step 1: Count input tokens per method per row using AI_COUNT_TOKENS ---
# NOTE: This is an illustrative ESTIMATE. The V2 prompt/model below are a representative
# abbreviation of an optimized body. Paste your actual OPTIMIZED_BODY prompt and winning
# MODEL from the deploy_v2 cell for an exact estimate.
token_sql = """
WITH tickets AS (
  SELECT TICKET_TEXT FROM VHOL_AI_DB.AI_FUNC.SUPPORT_TICKETS
),
-- Method 1: AI_CLASSIFY (built-in)
classify_tokens AS (
  SELECT 'EVAL_BUILTIN_CLASSIFY' AS method, 'AI_CLASSIFY' AS model,
         SUM(AI_COUNT_TOKENS('ai_classify', TICKET_TEXT,
             ['BILLING','TECHNICAL','ACCOUNT_ACCESS','FEATURE_REQUEST','CANCELLATION'])) AS input_tokens
  FROM tickets
),
-- Method 2: TRIAGE_TICKET V1 (llama3.1-8b, short prompt)
v1_tokens AS (
  SELECT 'EVAL_TRIAGE_CUSTOM_V1' AS method, 'llama3.1-8b' AS model,
         SUM(AI_COUNT_TOKENS('ai_complete', 'llama3.1-8b',
             'You are a customer support triage assistant for a SaaS company. Classify the support ticket into exactly one category: BILLING, TECHNICAL, ACCOUNT_ACCESS, FEATURE_REQUEST, or CANCELLATION. Ticket: ' || TICKET_TEXT
         )) AS input_tokens
  FROM tickets
),
-- Method 3: TRIAGE_TICKET V2 (llama3.1-8b, long optimized prompt)
v2_tokens AS (
  SELECT 'EVAL_TRIAGE_V2' AS method, 'llama3.1-8b' AS model,
         SUM(AI_COUNT_TOKENS('ai_complete', 'llama3.1-8b',
             'You are a customer support triage assistant for a SaaS company. Classify the ticket into exactly ONE category using these precise definitions and priority order. PRIORITY ORDER (apply top-down; first match wins): 1. CANCELLATION 2. FEATURE_REQUEST 3. ACCOUNT_ACCESS 4. TECHNICAL 5. BILLING. Output only the single category label. ' || TICKET_TEXT
         )) AS input_tokens
  FROM tickets
)
SELECT * FROM classify_tokens
UNION ALL SELECT * FROM v1_tokens
UNION ALL SELECT * FROM v2_tokens
"""

tokens_df = session.sql(token_sql).to_pandas()

# --- Step 2: Apply credit-per-token rates from account usage history ---
# Example rates observed from SNOWFLAKE.ACCOUNT_USAGE.CORTEX_FUNCTIONS_QUERY_USAGE_HISTORY.
# Check the Snowflake Service Consumption Table for current pricing in your region.
credit_rates = {
    'AI_CLASSIFY':  0.00000139,   # CLASSIFY_TEXT rate
    'llama3.1-8b':  0.00000019,   # COMPLETE llama3.1-8b rate
}

tokens_df['CREDITS_PER_TOKEN'] = tokens_df['MODEL'].map(credit_rates)
# Estimate ~15 output tokens per row for classification response, 40 rows
output_tokens_est = 15 * 40
tokens_df['TOTAL_TOKENS_EST'] = tokens_df['INPUT_TOKENS'] + output_tokens_est
tokens_df['EST_CREDITS'] = tokens_df['TOTAL_TOKENS_EST'] * tokens_df['CREDITS_PER_TOKEN']

print(tokens_df[['METHOD', 'MODEL', 'INPUT_TOKENS', 'TOTAL_TOKENS_EST', 'EST_CREDITS']].to_string(index=False))

# --- Step 3: Merge with quality scores from the compare cell ---
compare_df = compare.to_pandas() if hasattr(compare, 'to_pandas') else compare.copy()
compare_agg = compare_df.dropna(subset=['EXACT_MATCH']).groupby('EXPERIMENT').agg(
    EXACT_MATCH=('EXACT_MATCH', 'mean')
).reset_index()

plot_df = compare_agg.merge(tokens_df[['METHOD', 'MODEL', 'EST_CREDITS']],
                            left_on='EXPERIMENT', right_on='METHOD')
plot_df = plot_df.sort_values('EXACT_MATCH', ascending=True)

# --- Step 4: Dual-axis chart ---
labels = plot_df['EXPERIMENT'] + '\n(' + plot_df['MODEL'] + ')'
scores = plot_df['EXACT_MATCH'].astype(float)
costs  = plot_df['EST_CREDITS'].astype(float)

palette = {'EVAL_BUILTIN_CLASSIFY': '#A8A8A8', 'EVAL_TRIAGE_CUSTOM_V1': '#FF6F61', 'EVAL_TRIAGE_V2': '#29B5E8'}
colors = [palette.get(e, '#6BCB77') for e in plot_df['EXPERIMENT']]

fig, ax1 = plt.subplots(figsize=(10, 3.5))

# Quality bars
bars = ax1.barh(range(len(labels)), scores, color=colors, edgecolor='white', height=0.5, alpha=0.85)
ax1.set_yticks(range(len(labels)))
ax1.set_yticklabels(labels, fontsize=9)
ax1.set_xlabel('Exact Match Score', fontsize=10)
ax1.set_xlim(0, 1.15)
for bar, val in zip(bars, scores):
    ax1.text(val + 0.01, bar.get_y() + bar.get_height() / 2, f'{val:.1%}', va='center', fontsize=10)

# Cost dots (top axis)
ax2 = ax1.twiny()
ax2.plot(costs, range(len(labels)), 'D', color='#FF9800', markersize=9, zorder=5)
ax2.set_xlabel('Estimated Credits (40-row eval)', fontsize=10, color='#FF9800')
ax2.tick_params(axis='x', colors='#FF9800')
for cost, y in zip(costs, range(len(labels))):
    ax2.annotate(f'{cost:.4f}', (cost, y), textcoords='offset points',
                 xytext=(10, 0), va='center', fontsize=9, color='black', fontweight='bold')

ax1.set_title('Evaluation Comparison: Quality vs Estimated Cost', fontsize=12, pad=30)
fig.tight_layout()
plt.show()

## Recap
| Object | What it does |
|---|---|
| `CREATE AI FUNCTION` | Versionable, optimizable wrapper around `AI_COMPLETE` |
| `DATASET ... ADD VERSION` | Immutable labeled snapshot for experiments |
| `EXPERIMENT TYPE='AI_FUNCTION_EVALUATION'` | Scores any AI expression (built-in or custom) with a metric |
| `EXPERIMENT TYPE='AI_FUNCTION_OPTIMIZATION'` | Auto-tunes body + compares models on cost/quality |
| `SHOW RUNS / RUN METRICS / RUN PARAMETERS` | Read results |

**Cleanup:** `DROP DATABASE VHOL_AI_DB;`